# 13장 일반화와 모델 고르기 — 규제, 드롭아웃, 조기 종료, 체크포인트, 교차 검증, 종합 실습
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 과적합하는 신경망 만들기
2. L2 규제
3. 드롭아웃
4. 조기 종료와 체크포인트
5. 교차 검증으로 신경망, 로지스틱 회귀, 랜덤 포레스트 비교
6. 종합 실습: 캘리포니아 집값 예측

위에서부터 차례로 실행하세요. 학습에 시간이 걸리는 셀이 있습니다(5절과 6절은 각각 1분 안팎).

In [ ]:
# 텐서플로가 찍는 안내 메시지를 줄인다(결과에는 영향 없음)
import os, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)

## 1. 과적합하는 신경망
반달 데이터에서 학습 데이터를 60개만 쓰고, 은닉층 세 개(뉴런 256개씩)의 큰 신경망을 1,000 에포크 학습합니다. `run()`은 학습 정확도, 테스트 정확도, 테스트 로그 손실, 가중치 제곱합을 출력합니다.

In [ ]:
import numpy as np
import keras
from keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

X, y = make_moons(n_samples=1060, noise=0.3, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(
    X, y, train_size=60, random_state=0, stratify=y)

def run(model):                        # 학습한 뒤 성적을 출력한다
    opt = keras.optimizers.Adam(learning_rate=0.01)
    model.compile(optimizer=opt, loss="binary_crossentropy",
                  metrics=["accuracy"])
    model.fit(Xtr, ytr, epochs=1000, batch_size=60, verbose=0)
    tr = model.evaluate(Xtr, ytr, verbose=0)[1]
    te = model.evaluate(Xte, yte, verbose=0)[1]
    ll = log_loss(yte, model.predict(Xte, verbose=0).ravel())
    W = model.get_weights()[::2]       # 층마다의 가중치
    w2 = sum(float((w ** 2).sum()) for w in W)
    print(f"학습 정확도 {tr:.1%}, 테스트 정확도 {te:.1%}")
    print(f"테스트 로그 손실 {ll:.3f}, 가중치 제곱합 {w2:.0f}")

keras.utils.set_random_seed(0)
big = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
print("파라미터 수:", big.count_params())
run(big)

## 2. L2 규제
은닉층에 `kernel_regularizer=regularizers.l2(0.01)`을 걸어 가중치가 커지는 것에 벌점을 줍니다(λ = 0.01).

In [ ]:
from keras import regularizers

keras.utils.set_random_seed(0)
l2 = regularizers.l2(0.01)             # λ = 0.01
reg = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(256, activation="relu", kernel_regularizer=l2),
    layers.Dense(256, activation="relu", kernel_regularizer=l2),
    layers.Dense(256, activation="relu", kernel_regularizer=l2),
    layers.Dense(1, activation="sigmoid"),
])
run(reg)

## 3. 드롭아웃
은닉층 뒤에 `Dropout(0.5)`를 넣어 학습할 때 뉴런의 절반을 무작위로 끕니다. 예측할 때는 모든 뉴런을 씁니다.

In [ ]:
keras.utils.set_random_seed(0)
drop = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),               # 뉴런의 절반을 무작위로 끈다
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
])
run(drop)

세 신경망의 경계를 그려 봅니다(이북 그림 13.2, 13.3).

In [ ]:
import matplotlib.pyplot as plt

def plot_boundary(model, ax, title):
    g1, g2 = np.meshgrid(np.linspace(-1.7, 2.7, 200), np.linspace(-1.3, 1.8, 160))
    p = model.predict(np.c_[g1.ravel(), g2.ravel()], verbose=0).reshape(g1.shape)
    ax.contourf(g1, g2, p > 0.5, alpha=0.3, cmap="coolwarm")
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap="coolwarm", s=15, edgecolor="k", lw=0.3)
    ax.set_title(title)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.6))
plot_boundary(big, axs[0], "no regularization")
plot_boundary(reg, axs[1], "L2 (0.01)")
plot_boundary(drop, axs[2], "dropout 0.5")
plt.show()

## 4. 조기 종료와 체크포인트
12장의 유방암 신경망을 `EarlyStopping`(patience 20)과 `ModelCheckpoint`와 함께 최대 300 에포크 학습합니다. 12장에서 300 에포크까지 학습한 모델은 테스트 손실 0.186, 100 에포크에서 멈춘 모델은 0.081이었습니다.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

cancer = load_breast_cancer(as_frame=True)     # 12장과 같은 데이터
X = cancer.data.values
y = (cancer.target == 0).astype(int).values
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

def make_model():                       # 12장의 30-16-1 신경망
    m = keras.Sequential([
        keras.Input(shape=(30,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy"])
    return m

stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True)
save = keras.callbacks.ModelCheckpoint(
    "best.keras", monitor="val_loss", save_best_only=True)

keras.utils.set_random_seed(42)
model = make_model()
H = model.fit(Xtr_s, ytr, epochs=300, batch_size=32,
              validation_split=0.2, verbose=0,
              callbacks=[stop, save]).history
print("실제로 학습한 에포크:", len(H["loss"]))
best_ep = int(np.argmin(H["val_loss"])) + 1
print("검증 손실이 가장 낮은 에포크:", best_ep)
loss, acc = model.evaluate(Xte_s, yte, verbose=0)
print(f"테스트 손실 {loss:.3f}, 테스트 정확도 {acc:.1%}")

학습 곡선을 그려 봅니다(이북 그림 13.4). 점선 세로선이 검증 손실이 가장 낮은 에포크입니다.

In [ ]:
plt.plot(range(1, len(H["loss"]) + 1), H["loss"], label="train loss")
plt.plot(range(1, len(H["val_loss"]) + 1), H["val_loss"], "--", label="validation loss")
plt.axvline(best_ep, color="gray", ls=":")
plt.ylim(0, 0.35); plt.xlabel("epoch"); plt.legend(); plt.show()

저장한 체크포인트 파일(`best.keras`)을 불러 평가합니다. Colab 왼쪽의 파일 목록에서 파일을 볼 수 있습니다.

In [ ]:
best = keras.models.load_model("best.keras")  # 불러오기
loss, acc = best.evaluate(Xte_s, yte, verbose=0)
print(f"불러온 모델: 테스트 손실 {loss:.3f}, 정확도 {acc:.1%}")

## 5. 교차 검증으로 모델 비교하기
신경망을 5-폴드 교차 검증으로 평가합니다. 표준화는 폴드마다 학습 부분으로만 계산하고, 신경망은 폴드마다 새로 만듭니다.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for k, (a, b) in enumerate(skf.split(Xtr, ytr), start=1):
    sc_k = StandardScaler().fit(Xtr[a])  # 이 폴드의 학습 부분만
    keras.utils.set_random_seed(42)
    m = make_model()
    m.fit(sc_k.transform(Xtr[a]), ytr[a], epochs=100,
          batch_size=32, verbose=0)
    acc = m.evaluate(sc_k.transform(Xtr[b]), ytr[b], verbose=0)[1]
    scores.append(acc)
    print(f"폴드 {k}: 학습 {len(a)}건, 검증 {len(b)}건, "
          f"정확도 {acc:.1%}")
print(f"평균 {np.mean(scores):.1%}, 표준편차 {np.std(scores):.1%}")

In [ ]:
plt.bar(range(1, 6), scores)
plt.ylim(0.9, 1.01); plt.xlabel("fold"); plt.ylabel("validation accuracy"); plt.show()

같은 폴드로 로지스틱 회귀와 랜덤 포레스트를 평가합니다.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "로지스틱 회귀": make_pipeline(StandardScaler(),
                                   LogisticRegression()),
    "랜덤 포레스트": RandomForestClassifier(random_state=42),
}
for name, est in models.items():
    s = cross_val_score(est, Xtr, ytr, cv=skf)  # 같은 다섯 폴드
    print(f"{name}: 평균 {s.mean():.1%}, 표준편차 {s.std():.1%}")

## 6. 종합 실습: 캘리포니아 집값 예측
### 6-1. 데이터 살펴보기
처음 실행할 때 인터넷에서 데이터를 내려받습니다.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.shape)
print(df[["MedInc", "HouseAge", "AveOccup", "MedHouseVal"]]
      .describe().round(2))

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(df.Longitude, df.Latitude, c=df.MedHouseVal, s=1, cmap="viridis")
plt.colorbar(label="MedHouseVal (100k USD)"); plt.xlabel("longitude"); plt.ylabel("latitude")
plt.show()

### 6-2. 학습, 검증, 테스트로 나누기

In [ ]:
X = housing.data.values
y = housing.target.values
Xrest, Xte, yrest, yte = train_test_split(       # 테스트 20%
    X, y, test_size=0.2, random_state=42)
Xtr, Xva, ytr, yva = train_test_split(           # 나머지의 25%
    Xrest, yrest, test_size=0.25, random_state=42)
print("학습", len(Xtr), "검증", len(Xva), "테스트", len(Xte))

### 6-3. 후보 모델 비교(검증 데이터)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

def report(name, pred):                          # 검증 데이터 성적
    rmse = mean_squared_error(yva, pred) ** 0.5
    r2 = r2_score(yva, pred)
    print(f"{name}: RMSE {rmse:.3f}, R² {r2:.3f}")

cands = {
    "선형 회귀": make_pipeline(StandardScaler(),
                               LinearRegression()),
    "랜덤 포레스트": RandomForestRegressor(random_state=42,
                                          n_jobs=-1),
    "그레이디언트 부스팅":
        HistGradientBoostingRegressor(random_state=42),
}
for name, est in cands.items():
    est.fit(Xtr, ytr)
    report(name, est.predict(Xva))

신경망에 조기 종료를 붙여 학습합니다.

In [ ]:
sh = StandardScaler().fit(Xtr)
keras.utils.set_random_seed(42)
net = keras.Sequential([
    keras.Input(shape=(8,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1),
])
net.compile(optimizer="adam", loss="mse")
stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True)
H = net.fit(sh.transform(Xtr), ytr, epochs=200, batch_size=32,
            validation_data=(sh.transform(Xva), yva),
            verbose=0, callbacks=[stop]).history
print("학습한 에포크:", len(H["loss"]))
report("신경망", net.predict(sh.transform(Xva), verbose=0).ravel())

### 6-4. 고른 모델을 테스트 데이터로 한 번

In [ ]:
final = HistGradientBoostingRegressor(random_state=42)
final.fit(Xrest, yrest)                     # 학습 + 검증 데이터
pred = final.predict(Xte)
rmse = mean_squared_error(yte, pred) ** 0.5
print(f"테스트 RMSE {rmse:.3f}, R² {r2_score(yte, pred):.3f}")

---
## 실습 문제

**문제 1.** 2절의 L2 규제 강도를 0.001, 0.1로 바꾸고, `regularizers.l1(0.001)`로 L1 규제도 해 보세요. 학습 정확도, 테스트 정확도, 가중치 제곱합을 2절(λ = 0.01)과 비교하면 어떤가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 3절의 드롭아웃 비율을 0.2와 0.8로 바꾸어 학습 정확도, 테스트 정확도, 테스트 로그 손실을 0.5와 비교하세요. 비율이 너무 크면 어떻게 되나요?

In [ ]:
# 문제 2 풀이

**문제 3.** 4절의 조기 종료에서 `patience`를 5와 50으로 바꾸어 실제로 학습한 에포크 수와 테스트 손실을 비교하세요. 또 `patience=20`에서 `restore_best_weights=False`로 바꾸면 테스트 손실이 어떻게 되나요?

In [ ]:
# 문제 3 풀이

**문제 4.** 6절의 신경망에서 (가) 은닉층 뒤에 `Dropout(0.2)`를 넣은 경우, (나) 은닉 뉴런을 128개씩으로 늘린 경우의 검증 RMSE를 구해 0.514와 비교하세요. 그레이디언트 부스팅(0.477)을 따라잡았나요?

In [ ]:
# 문제 4 풀이